In [94]:
import pandas as pd
import time
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import requests
import re
from rapidfuzz import fuzz, process


In [95]:
ObservationsData = pd.read_csv(r'../Transformed Data/ObservationsClean.csv', sep=';', index_col=0)
NativeData = pd.read_csv(r'../Transformed Data/NativeDataClean.csv', sep=';', index_col=0)

In [96]:
References = pd.DataFrame()
References['Reference'] = pd.concat([ObservationsData['Reference'], NativeData['Reference']], ignore_index=True)
References.drop_duplicates(inplace=True)
References.reset_index(drop=True, inplace=True)

# Code

In [59]:
def TitleExtract(references):
    
    def CleanRefs(reference_list):
        def Text(text):
            return re.sub(r"[^a-zA-Z0-9.,() \-]", "", text)
        return [Text(ref) for ref in reference_list]

    def Title(ref):
        TitleAfterYear = re.search(r'\(\d{4}\)\.\s*(.*?)(?:\. [A-Z]|\.$)', ref)
        if TitleAfterYear:
            return TitleAfterYear.group(1).strip()

        SentenceAfterYear = re.search(r'\(\d{4}\)\.\s*(.*)', ref)
        if SentenceAfterYear:
            return SentenceAfterYear.group(1).strip()
    
        return None
    
    references = CleanRefs(references)
    return [Title(ref) for ref in references]

In [60]:
nest_asyncio.apply()
@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))

def Extract_CrossRef(titles):
    
    async def CrossRef_Sessions(titles):
        
        async def Crossref_Search(session, title):
            
            url = "https://api.crossref.org/works"
            params = {"query.title": title, "rows": 1}
    
            def CleanText(text):
                return re.sub(r"[^a-zA-Z0-9.,() \-]", "", text)
    
            try:
                async with session.get(url, params=params) as response:
                    response.raise_for_status()
                    data = await response.json()
                    item = data['message']['items'][0]
            
                    authors = ', '.join([CleanText(f"{a.get('family', '').title()}, {
                        a.get('given', '')[:1].upper()}.").strip() for a in item.get(
                            'author', [])]) or "Unknown Author"
                    year = item.get('issued', {}).get('date-parts', [[None]])[0][0]
                    title_clean = CleanText(item.get('title', [''])[0]).title()
                    doi = item.get('DOI')
            
                    if item.get("type") == "journal-article":
                        journal = CleanText(item.get('container-title', [''])[0])
                        volume = CleanText(item.get('volume', ''))
                        pages = CleanText(item.get('page', ''))
                        
                        return f"{authors} ({year}). {title_clean}. {journal}, {volume}, {pages}. DOI: {doi}"
                        
                    if item.get("type") == "book-chapter":
                        book = CleanText(item.get('container-title', [''])[0])
                        pages = CleanText(item.get('page', ''))
                        
                        return f"{authors} ({year}). {title_clean}. In: {book}, {pages}. DOI: {doi}"
                    
                    else: 
                        journal == None, volume == None, pages == None
                        return f"{authors} ({year}). {title_clean}. DOI: {doi}"
            
                    
    
            except Exception as e:
                return None

        async with aiohttp.ClientSession() as session:
            tasks = [Crossref_Search(session, title) for title in titles]
            return await asyncio.gather(*tasks)
    return asyncio.get_event_loop().run_until_complete(CrossRef_Sessions(titles))



# Extraction

In [20]:
References['Title'] = TitleExtract(References['Reference'])

In [21]:
References['CrossRef'] = Extract_CrossRef(References['Title'])

## Error Corrections

In [22]:
len(References[References['CrossRef'].str.contains("Unknown Author", na=False)]['Reference'])


49

In [23]:
len(References[References['CrossRef'].isna()]['Reference'])

134

In [97]:
References.to_csv(r'../Transformed Data/ReferencesClean.csv', sep=';', index=False)

In [25]:
References = pd.read_csv(r'../Transformed Data/ReferencesClean.csv', sep=';')

In [26]:
References

,Reference,Title,CrossRef
0,CAB International (CABI) (2024). CABI Invasive...,CABI Invasive Species Compendium,"Diaz-Soltero, H., Scott, P. (2014). Global Ide..."
1,"Gilligan, T., Brown, J. and Baixeras, J. (2020...",Immigrant Tortricidae Holarctic versus Introdu...,"Gilligan, T., Brown, J., Baixeras, J. (2020). ..."
2,European and Mediterranean Plant Protection Or...,EPPO Global Database,"Roy, A., Petter, F., Griessinger, D. (2010). E..."
3,"Baker, A. and Stiling, P. (2009). Comparing th...",Comparing the effects of the exotic cactus-fee...,"Baker, A., Stiling, P. (2008). Comparing The E..."
4,U.S. Geological Survey (2022). United States R...,United States Register of Introduced and Invas...,NaN
...,...,...,...
1173,"Bidzilya, O., Karsholt, O., Kravchenko, V. and...",An annotated checklist of Gelechiidae (Lepidop...,"Bidzilya, O., Karsholt, O., Kravchenko, V., um..."
1174,"Pogue, M. (2013). Revised status of Chloridea ...",Revised status of Chloridea Duncan and (Westwo...,"Pogue, M. (2013). Revised Status Ofichlorideai..."
1175,"Garre, M., Girdley, J., Guerrero, J., Rubio, R...",An annotated checklist of the Crambidae of the...,"Garre, M., Girdley, J., Guerrero, J., Rubio, R..."
1176,"Withers, T. (2001). Colonization of eucalypts ...",Colonization of eucalypts in New Zealand by Au...,"Withers, T. (2001). Colonization Of Eucalypts ..."


In [27]:
References[References['CrossRef'].isna()]['Reference']

4       U.S. Geological Survey (2022). United States R...
6       Shine, C., Reaser, J.  and Gutierrez, A. (eds....
33      Sandoval, A., Ide, S., Rothmann, S., Zuniga, E...
34      Andreas, J., Price, J. and Grevstad, F. (2022)...
61      Braby, M., Bertelsmeier, C., Sanderson, C. and...
                              ...                        
1123    Adams, J. (1992). A new lichen moth record for...
1137    Howard, L. and Chittenden, F. (1909). The Leop...
1140    De-Gregorio, J. (1982). Las Platyperigea Smith...
1142    Gomboc, S. and Koren, T. (2015). The distribut...
1177    Karsholt, O., Baldizzone, G. and Gomboc, S. (2...
Name: Reference, Length: 134, dtype: object

In [28]:
References[References['CrossRef'].str.contains("Unknown Author", na=False)]['Reference']

13      Invasive Species Specialist Group ISSG (2015)....
28      Warren, L. and Tadic, M. (1970). Fall web worm...
40      National Plant Protection Organizations within...
73      Meagher, R., Brambila, J. and Hung, E. (2008)....
105     Liu T., Cai Y., Wang C. and Li H. (2015). Biol...
168     European and Mediterranean Plant Protection Or...
183     European and Mediterranean Plant Protection Or...
220     Timus, A. (2015). The Invasive Entomofauna of ...
233     Czerniakowski, Z. and Olbrycht, T. (2015). Inv...
278     European and Mediterranean Plant Protection Or...
286     du Plessis, H. (2002). Groundnut leaf miner, A...
298     Way, M. and Turner, P. (1999). The spotted bor...
304     Badawy, A. (1967). The morphology and biology ...
327     Naik, S., Jayashankar, M. and Sridhar, V. (201...
329     Deshmukh, S., Kalleshwaraswamy, C., Asokan, R....
351     Williams, J. (1983). The sugar cane stem borer...
354     Guerout, R. (1974). Occurrence of Phyllocnisti...
362     van Ni